# 04-01 어텐션은 왜 필요한가

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 임베딩만으로는 문맥을 모르는 문제

In [ ]:
vocab = ["<pad>", "배", "가", "아프다", "고프다", "바다", "에", "떠", "있다", "맛있다"]
stoi = {t: i for i, t in enumerate(vocab)}
emb = nn.Embedding(len(vocab), 4)

s1 = ["배", "가", "고프다"]
s2 = ["바다", "에", "배", "가", "떠", "있다"]
v1 = emb(torch.tensor([stoi[t] for t in s1]))  # (3, 4)
v2 = emb(torch.tensor([stoi[t] for t in s2]))  # (6, 4)
print(torch.equal(v1[0], v2[2]))  # 두 문장의 "배"가 같은 벡터

## 2. 아이디어: 다른 토큰을 섞은 가중 평균

In [ ]:
torch.manual_seed(0)
x = torch.randn(4, 8)  # (seq_len=4, d=8), 토큰 4개

weights = torch.tensor([0.1, 0.6, 0.1, 0.2])  # 합이 1인 가중치
mixed = (weights.unsqueeze(1) * x).sum(dim=0)  # (8,)
print(mixed.shape)
print(torch.allclose(mixed, weights @ x))  # 행렬 곱과 같아요

## 3. 가중치는 어떻게 정할까: 쿼리, 키, 값

In [ ]:
d = 8
torch.manual_seed(0)
q_proj = nn.Linear(d, d, bias=False)
k_proj = nn.Linear(d, d, bias=False)
v_proj = nn.Linear(d, d, bias=False)

Q = q_proj(x)  # (4, 8), 내가 찾는 것
K = k_proj(x)  # (4, 8), 내가 가진 특징
V = v_proj(x)  # (4, 8), 내가 전할 내용

scores = Q @ K.T / math.sqrt(d)  # (4, 4), 토큰 i가 토큰 j를 얼마나 볼지
attn = F.softmax(scores, dim=-1)  # (4, 4), 행마다 합이 1
out = attn @ V  # (4, 8)
print(attn)
print(attn.sum(dim=-1))
print(out.shape)

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.imshow(attn.detach(), cmap="Oranges", vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        plt.text(j, i, f"{attn[i, j]:.2f}", ha="center", va="center")
plt.xlabel("key position (j)")
plt.ylabel("query position (i)")
plt.title("attention weights")
plt.colorbar()
plt.show()

## 4. 함수로 묶고 PyTorch와 비교하기

In [ ]:
def attention(Q, K, V):
    # Q, K, V: (..., seq_len, d)
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)  # (..., seq_len, seq_len)
    weights = F.softmax(scores, dim=-1)  # (..., seq_len, seq_len)
    return weights @ V, weights  # (..., seq_len, d)

out_mine, _ = attention(Q, K, V)
out_torch = F.scaled_dot_product_attention(Q, K, V)
print(torch.allclose(out_mine, out_torch, atol=1e-6))

## 5. 왜 √d로 나눌까

In [ ]:
torch.manual_seed(0)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, d_ in zip(axes, [8, 64, 512]):
    q = torch.randn(d_)
    k = torch.randn(10, d_)  # 키 10개
    raw = F.softmax(k @ q, dim=-1)
    scaled = F.softmax(k @ q / math.sqrt(d_), dim=-1)
    ax.bar(torch.arange(10) - 0.2, raw, width=0.4, label="no scaling", color="tab:gray")
    ax.bar(torch.arange(10) + 0.2, scaled, width=0.4, label="/ sqrt(d)", color="tab:orange")
    ax.set_title(f"d = {d_}")
    ax.set_ylim(0, 1)
axes[0].legend()
plt.show()

In [ ]:
for d_ in [8, 64, 512]:
    torch.manual_seed(0)
    q, k = torch.randn(1000, d_), torch.randn(1000, d_)
    dots = (q * k).sum(-1)  # (1000,)
    print(f"d={d_:3d}: 내적의 표준편차 {dots.std():.1f}, sqrt(d)로 나누면 {(dots / math.sqrt(d_)).std():.2f}")